In [ ]:
# Cell 1 — Imports and configuration (unchanged from original)
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'config.py').exists() and (candidate / 'evaluation' / 'test_questions.json').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vector_rag.pipeline     import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline
from hybrid_rag.pipeline     import HybridRAGPipeline
from evaluation.evaluator    import run_evaluation, print_summary
from evaluation.ragas_evaluator import run_ragas_evaluation, merge_results, print_ragas_summary

RESULTS_DIR    = REPO_ROOT / 'evaluation' / 'results'
QUESTIONS_PATH = REPO_ROOT / 'evaluation' / 'test_questions.json'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

METHOD_ORDER  = ['vector', 'vectorless', 'hybrid']
METHOD_LABELS = {'vector': 'Vector RAG', 'vectorless': 'Vectorless RAG', 'hybrid': 'Hybrid RAG'}
METHOD_COLORS = {'vector': '#1f77b4', 'vectorless': '#ff7f0e', 'hybrid': '#2ca02c'}
RAGAS_METRIC_LABELS = {
    'answer_relevancy'    : 'Answer Relevancy',
    'faithfulness'        : 'Faithfulness',
    'context_precision'   : 'Contextual Precision',
    'context_recall'      : 'Contextual Recall',
    'contextual_relevancy': 'Contextual Relevancy\n(≈ context_precision)',
}

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 160,
                     'axes.titlesize': 13, 'axes.labelsize': 11})

def save_figure(fig, filename):
    out = RESULTS_DIR / filename
    fig.savefig(out, dpi=160, bbox_inches='tight')
    return out

print('Imports ready.')

In [ ]:
# Cell 2 — Initialise pipelines
with open(QUESTIONS_PATH) as f:
    _q = json.load(f)
print(f'Loaded {len(_q["questions"])} questions for the benchmark.')

print('Initialising pipelines...')
vec    = VectorRAGPipeline()
vl     = VectorlessRAGPipeline()
hybrid = HybridRAGPipeline()
print('All three pipelines are ready.')

In [ ]:
# Cell 3 — Run judge evaluation
#
# capture_contexts=True tells run_evaluation() to also return a dict
# {(question_id, method): [context_str, ...]} so RAGAS can reuse the
# same retrieved contexts without re-running retrieval.
#
# The judge CSV is saved to: evaluation/results/three_way_judge_results.csv

judge_df, retrieved_contexts_map = run_evaluation(
    vector_pipeline     = vec,
    vectorless_pipeline = vl,
    hybrid_pipeline     = hybrid,
    results_filename    = 'three_way_judge_results.csv',
    capture_contexts    = True,          # NEW — required for RAGAS
)

print(f'Judge evaluation complete — {len(judge_df)} rows')
print_summary(judge_df)

In [ ]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from evaluation.ragas_evaluator import run_ragas_evaluation
print("judge_df" in globals())
print("retrieved_contexts_map" in globals())

In [1]:
# Cell 4 — Run RAGAS evaluation
#
# RAGAS uses the Mistral judge API via an OpenAI-compatible wrapper.
# Make sure MISTRAL_JUDGE_API_KEY is set in your .env.
#
# Metrics computed:
#   answer_relevancy     → ResponseRelevancy: is the answer on-topic?
#   faithfulness         → Is every claim grounded in retrieved context?
#   context_precision    → Are the best chunks ranked first?
#   context_recall       → Does context cover the reference answer?
#                          (NaN when no reference_answer in test_questions.json)
#   contextual_relevancy → Same value as context_precision
#                          (no standalone metric in RAGAS 0.2.x)
#
# The RAGAS CSV is saved to: evaluation/results/three_way_ragas_results.csv
#
# RATE LIMIT TIP: batch_size=5 is safe with Mistral free-tier (1 RPM for judges).
# Increase to 10-15 if you have a higher-tier key.

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from evaluation.ragas_evaluator import run_ragas_evaluation
assert 'judge_df' in globals() and 'retrieved_contexts_map' in globals(), (
    "judge_df and retrieved_contexts_map must be defined first. "
    "Run the judge evaluation cell above before this RAGAS cell."
)
ragas_df = run_ragas_evaluation(
    judge_df               = judge_df,
    retrieved_contexts_map = retrieved_contexts_map,
    questions_path         = QUESTIONS_PATH,
    results_filename       = 'three_way_ragas_results.csv',
    batch_size             = 5,
)

print(f'RAGAS evaluation complete — {len(ragas_df)} rows')
print_ragas_summary(ragas_df)

AssertionError: judge_df and retrieved_contexts_map must be defined first. Run the judge evaluation cell above before this RAGAS cell.

In [ ]:
# import time

# # Create a unique filename for this run
# timestamp = time.strftime("%Y%m%d-%H%M%S")
# new_filename = f"three_way_ragas_results_{timestamp}.csv"
  
# ragas_df = run_ragas_evaluation(
#     judge_df               = judge_df,
#     retrieved_contexts_map = retrieved_contexts_map,
#     questions_path         = QUESTIONS_PATH,
#     results_filename       = new_filename  # Use the unique name
# )